# Day10：长期 Memory

本 Notebook 验证项目隔离的四类长期记忆，并演示一次已通过验证的 CS0246 namespace 修复如何指导后续诊断。生产实现位于 `memory/long_term.py` 和 `workflow/long_term_memory.py`。

In [ ]:
import os
import tempfile
from pprint import pprint

from memory.long_term import LongTermMemoryStore
from workflow.long_term_memory import LongTermMemoryNode

temporary_directory = tempfile.TemporaryDirectory()
project_path = os.path.join(temporary_directory.name, 'UnityProject')
store = LongTermMemoryStore(os.path.join(temporary_directory.name, 'memory.json'))
node = LongTermMemoryNode(store, project_path)

In [ ]:
node.update_project({
    'project_context': {
        'schema_version': 1,
        'project': {'name': 'Inventory'},
        'summary': {'scripts': 5},
    }
})
store.remember_coding_style(project_path, 'Use the InventorySystem namespace')
project_memory = store.get_project(project_path)
assert set(project_memory) == {
    'project_memory', 'coding_style', 'bug_history', 'solution_history'
}
pprint(project_memory)

In [ ]:
compile_error = {
    'code': 'CS0246',
    'file': 'InventoryManager.cs',
    'message': "The type or namespace name 'ItemData' could not be found",
}
node.observe_compile({
    'compile_result': {'success': False, 'system_error': False, 'errors': [compile_error]},
    'repair_history': [],
})
repair = {
    'round': 1,
    'status': 'success',
    'actions': [{
        'success': True,
        'root': {
            'error_code': 'CS0246',
            'cause': 'missing_using',
            'fix_strategy': 'Check the namespace before creating a duplicate type',
            'fix_action': {'operation': 'add_using'},
        },
    }],
}
node.observe_compile({
    'compile_result': {'success': True, 'system_error': False, 'errors': []},
    'repair_history': [repair],
})

In [ ]:
recalled = node.observe_compile({
    'compile_result': {'success': False, 'system_error': False, 'errors': [compile_error]},
    'repair_history': [],
})['memory_context']
assert recalled['insights'][0]['error_code'] == 'CS0246'
assert recalled['insights'][0]['successful_operations'] == ['add_using']
pprint(recalled)
temporary_directory.cleanup()